In [ ]:
# 10.0：最终研究报告环境初始化与证据完整性检查 ----
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"
DOCS_DIR = PROJECT_ROOT / "docs"

REQUIRED_EVIDENCE = {
    "综合因子有效性": (
        TABLE_DIR / "05_integrated_profitability_effectiveness_summary.csv"
    ),
    "HAC 与逐年稳定性": (
        TABLE_DIR / "09_integrated_profitability_hac_stability_summary.csv"
    ),
    "RQAlpha 三组合表现": (
        TABLE_DIR / "08_rqalpha_oos_three_portfolio_performance.csv"
    ),
    "RQAlpha 相对比较": (
        TABLE_DIR / "08_rqalpha_oos_q5_relative_comparison.csv"
    ),
    "强制清仓策略表现": (
        TABLE_DIR / "09_rqalpha_forced_exit_sensitivity_performance.csv"
    ),
    "强制清仓相对表现": (
        TABLE_DIR / "09_rqalpha_forced_exit_relative_comparison.csv"
    ),
    "最终决策日志": (
        TABLE_DIR / "09_final_research_decision_log.csv"
    ),
    "最终决策规则": (
        METADATA_DIR / "09_final_research_decision_rules.json"
    ),
    "研究协议": (
        DOCS_DIR / "09_RESEARCH_PROTOCOL.md"
    ),
}

missing_evidence = {
    name: path
    for name, path in REQUIRED_EVIDENCE.items()
    if not path.exists()
}

assert not missing_evidence, (
    "以下最终研究证据缺失：\n"
    + "\n".join(
        f"- {name}: {path}"
        for name, path in missing_evidence.items()
    )
)

print("===== 10 最终研究报告环境已启动 =====")
print("项目根目录：", PROJECT_ROOT)
print("已验证证据数：", len(REQUIRED_EVIDENCE))

for name, path in REQUIRED_EVIDENCE.items():
    print(f"✓ {name}：{path.name}")

print("\n下一步：10.1 读取冻结结果，生成最终报告的核心证据表。")

In [ ]:
# 10.1：读取冻结结果，生成最终报告核心证据表 ----
FACTOR_SUMMARY_PATH = (
    TABLE_DIR / "05_integrated_profitability_effectiveness_summary.csv"
)
HAC_PATH = (
    TABLE_DIR / "09_integrated_profitability_hac_stability_summary.csv"
)
ORIGINAL_STRATEGY_PATH = (
    TABLE_DIR / "08_rqalpha_oos_three_portfolio_performance.csv"
)
FORCED_PERFORMANCE_PATH = (
    TABLE_DIR / "09_rqalpha_forced_exit_sensitivity_performance.csv"
)
FORCED_RELATIVE_PATH = (
    TABLE_DIR / "09_rqalpha_forced_exit_relative_comparison.csv"
)
DECISION_LOG_PATH = (
    TABLE_DIR / "09_final_research_decision_log.csv"
)
DECISION_RULES_PATH = (
    METADATA_DIR / "09_final_research_decision_rules.json"
)

factor_summary = pd.read_csv(FACTOR_SUMMARY_PATH)
hac_summary = pd.read_csv(HAC_PATH)
original_strategy = pd.read_csv(ORIGINAL_STRATEGY_PATH)
forced_performance = pd.read_csv(FORCED_PERFORMANCE_PATH)
forced_relative = pd.read_csv(FORCED_RELATIVE_PATH)
decision_log = pd.read_csv(DECISION_LOG_PATH)

with open(DECISION_RULES_PATH, "r", encoding="utf-8") as file:
    decision_rules = json.load(file)

factor_oos = factor_summary.loc[
    (
        factor_summary["factor_name"]
        == "integrated_profitability_structure_proxy_strict"
    )
    & (
        factor_summary["period"]
        == "out_of_sample_2023_2025"
    )
].iloc[0]

hac_oos = hac_summary.loc[
    hac_summary["period"] == "out_of_sample_2023_2025"
].iloc[0]

q5_forced = forced_performance.loc[
    forced_performance["portfolio"] == "Q5_factor_high"
].iloc[0]

q1_forced = forced_performance.loc[
    forced_performance["portfolio"] == "Q1_factor_low"
].iloc[0]

equal_weight_forced = forced_performance.loc[
    forced_performance["portfolio"]
    == "Equal_weight_eligible_universe"
].iloc[0]

q5_vs_q1 = forced_relative.loc[
    forced_relative["comparison"]
    == "Q5_factor_high_vs_Q1_factor_low"
].iloc[0]

q5_vs_equal_weight = forced_relative.loc[
    forced_relative["comparison"]
    == "Q5_factor_high_vs_Equal_weight_eligible_universe"
].iloc[0]

final_decision = decision_rules["final_research_decision"]

core_evidence = pd.DataFrame(
    [
        {
            "section": "因子样本外检验",
            "metric": "样本外观测期数",
            "value": int(factor_oos["observations"]),
            "interpretation": "综合因子严格可用的样本外月度观测数。",
        },
        {
            "section": "因子样本外检验",
            "metric": "样本外平均 RankIC",
            "value": float(factor_oos["mean_rank_ic"]),
            "interpretation": "方向为正，但幅度较弱。",
        },
        {
            "section": "因子样本外检验",
            "metric": "样本外 RankIC HAC p 值",
            "value": float(hac_oos["rank_ic_hac_p_value"]),
            "interpretation": "未通过 5% 显著性门槛。",
        },
        {
            "section": "因子样本外检验",
            "metric": "样本外 Q5-Q1 平均收益",
            "value": float(
                factor_oos["mean_high_minus_low_return"]
            ),
            "interpretation": "方向为正，但需要结合 HAC 检验解释。",
        },
        {
            "section": "因子样本外检验",
            "metric": "样本外 Q5-Q1 HAC p 值",
            "value": float(
                hac_oos["high_minus_low_return_hac_p_value"]
            ),
            "interpretation": "未通过 5% 显著性门槛。",
        },
        {
            "section": "RQAlpha 排序证据",
            "metric": "Q5 相对 Q1 收益（强制清仓后）",
            "value": float(
                q5_vs_q1[
                    "relative_return_at_forced_exit_end"
                ]
            ),
            "interpretation": "保留高低分组正向排序证据。",
        },
        {
            "section": "RQAlpha 增量 Alpha",
            "metric": "Q5 相对等权券池收益（强制清仓后）",
            "value": float(
                q5_vs_equal_weight[
                    "relative_return_at_forced_exit_end"
                ]
            ),
            "interpretation": "未超过等权可投资券池。",
        },
        {
            "section": "执行稳健性",
            "metric": "三个组合均完成强制清仓",
            "value": bool(
                forced_performance[
                    "fully_liquidated_at_forced_exit_end"
                ]
                .astype(str)
                .str.lower()
                .eq("true")
                .all()
            ),
            "interpretation": "最终结论不依赖期末未成交头寸的市值计价。",
        },
        {
            "section": "最终决策",
            "metric": "研究状态",
            "value": final_decision,
            "interpretation": (
                "保留为研究候选，不认定为已确认独立 Alpha。"
            ),
        },
    ]
)

strategy_report_table = (
    original_strategy[
        [
            "portfolio",
            "total_return",
            "annualized_return",
            "sharpe_ratio",
            "information_ratio",
            "max_drawdown",
        ]
    ]
    .merge(
        forced_performance[
            [
                "portfolio",
                "forced_exit_final_total_return",
                "post_exit_tail_return",
                "final_market_value",
                "fully_liquidated_at_forced_exit_end",
                "post_exit_trade_record_count",
                "post_exit_transaction_cost",
            ]
        ],
        on="portfolio",
        how="left",
    )
    .sort_values("portfolio")
    .reset_index(drop=True)
)

strategy_report_table = strategy_report_table.rename(
    columns={
        "total_return": "original_rqalpha_total_return",
        "annualized_return": "original_rqalpha_annualized_return",
        "sharpe_ratio": "original_rqalpha_sharpe_ratio",
        "information_ratio": "original_rqalpha_information_ratio",
        "max_drawdown": "original_rqalpha_max_drawdown",
    }
)

core_evidence.to_csv(
    TABLE_DIR / "10_final_report_core_evidence.csv",
    index=False,
)
strategy_report_table.to_csv(
    TABLE_DIR / "10_final_report_strategy_comparison.csv",
    index=False,
)

report_context = {
    "report_scope": "frozen_research_results_only",
    "final_research_decision": final_decision,
    "core_evidence_file": str(
        TABLE_DIR / "10_final_report_core_evidence.csv"
    ),
    "strategy_comparison_file": str(
        TABLE_DIR / "10_final_report_strategy_comparison.csv"
    ),
    "source_protocol": str(DOCS_DIR / "09_RESEARCH_PROTOCOL.md"),
    "no_post_oos_optimization": True,
}

with open(
    METADATA_DIR / "10_final_report_context.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(report_context, file, ensure_ascii=False, indent=2)

print("===== 最终报告核心证据 =====")
display(core_evidence)

print("\n===== 策略与强制清仓对照 =====")
display(strategy_report_table)

print("\n===== 已保存 =====")
print(TABLE_DIR / "10_final_report_core_evidence.csv")
print(TABLE_DIR / "10_final_report_strategy_comparison.csv")
print(METADATA_DIR / "10_final_report_context.json")

In [ ]:
# 10.2：生成最终 Markdown 研究报告 ----
CORE_EVIDENCE_PATH = (
    TABLE_DIR / "10_final_report_core_evidence.csv"
)
STRATEGY_COMPARISON_PATH = (
    TABLE_DIR / "10_final_report_strategy_comparison.csv"
)
DECISION_LOG_PATH = (
    TABLE_DIR / "09_final_research_decision_log.csv"
)
REPORT_PATH = DOCS_DIR / "10_FINAL_RESEARCH_REPORT.md"

for path in [
    CORE_EVIDENCE_PATH,
    STRATEGY_COMPARISON_PATH,
    DECISION_LOG_PATH,
]:
    assert path.exists(), f"缺少最终报告输入：{path}"

core_evidence = pd.read_csv(CORE_EVIDENCE_PATH)
strategy_comparison = pd.read_csv(STRATEGY_COMPARISON_PATH)
decision_log = pd.read_csv(DECISION_LOG_PATH)

final_decision = core_evidence.loc[
    core_evidence["metric"] == "研究状态",
    "value",
].iloc[0]

rank_ic = float(
    core_evidence.loc[
        core_evidence["metric"] == "样本外平均 RankIC",
        "value",
    ].iloc[0]
)
rank_ic_p = float(
    core_evidence.loc[
        core_evidence["metric"] == "样本外 RankIC HAC p 值",
        "value",
    ].iloc[0]
)
high_low_return = float(
    core_evidence.loc[
        core_evidence["metric"] == "样本外 Q5-Q1 平均收益",
        "value",
    ].iloc[0]
)
high_low_p = float(
    core_evidence.loc[
        core_evidence["metric"] == "样本外 Q5-Q1 HAC p 值",
        "value",
    ].iloc[0]
)
q5_vs_q1 = float(
    core_evidence.loc[
        core_evidence["metric"] == "Q5 相对 Q1 收益（强制清仓后）",
        "value",
    ].iloc[0]
)
q5_vs_equal_weight = float(
    core_evidence.loc[
        core_evidence["metric"] == "Q5 相对等权券池收益（强制清仓后）",
        "value",
    ].iloc[0]
)

strategy_display = strategy_comparison.copy()

for column in [
    "original_rqalpha_total_return",
    "original_rqalpha_annualized_return",
    "original_rqalpha_max_drawdown",
    "forced_exit_final_total_return",
    "post_exit_tail_return",
]:
    strategy_display[column] = strategy_display[column].map(
        lambda value: f"{float(value):.2%}"
    )

for column in [
    "original_rqalpha_sharpe_ratio",
    "original_rqalpha_information_ratio",
]:
    strategy_display[column] = strategy_display[column].map(
        lambda value: f"{float(value):.3f}"
    )

strategy_display["final_market_value"] = strategy_display[
    "final_market_value"
].map(lambda value: f"{float(value):,.0f}")

strategy_display["post_exit_transaction_cost"] = strategy_display[
    "post_exit_transaction_cost"
].map(lambda value: f"{float(value):,.2f}")

strategy_display = strategy_display.rename(
    columns={
        "portfolio": "组合",
        "original_rqalpha_total_return": "原始累计收益",
        "original_rqalpha_annualized_return": "原始年化收益",
        "original_rqalpha_sharpe_ratio": "Sharpe",
        "original_rqalpha_information_ratio": "信息比率",
        "original_rqalpha_max_drawdown": "最大回撤",
        "forced_exit_final_total_return": "强制清仓后累计收益",
        "post_exit_tail_return": "退出尾部收益",
        "final_market_value": "最终残余市值",
        "fully_liquidated_at_forced_exit_end": "完成清仓",
        "post_exit_trade_record_count": "尾部成交笔数",
        "post_exit_transaction_cost": "尾部交易成本",
    }
)

decision_display = decision_log.rename(
    columns={
        "evidence": "证据",
        "finding": "发现",
        "decision_implication": "研究含义",
    }
)

report_text = f"""# A 股盈利能力因子透明复现研究：最终报告

## 一句话结论

本项目完成了基于研报四个底层因子的等权透明实现、样本外检验与带执行约束的 RQAlpha 回测。当前等权透明实现保留为研究候选，但未确认具有独立、稳定且超过等权可投资券池的增量选股 Alpha。该结论不验证或否定研报原始两层权重公式。

最终状态：`{final_decision}`。

## 1. 研究问题与 Alpha 假设

研究假设是：企业的可持续盈利能力、低盈利波动、较低杠杆扭曲与较高现金盈利质量，可能反映竞争优势、资本配置质量或市场对基本面持续性的定价滞后。

本项目检验的不是“策略是否在历史上赚钱”，而是该结构化盈利信号能否：

1. 对未来 20 个交易日收益形成稳定的横截面排序；
2. 使 Q5 高分组优于 Q1 低分组；
3. 在真实成本和交易约束下，超过不按因子排序的等权可投资券池。

## 2. 数据、样本与时间对齐

- 股票池：动态中证 800；
- 研究期：2016-01 至 2025-12；
- 样本内：2016-2022；
- 样本外：2023-2025 信号，最后持有窗口延长至 2026-02；
- 标签：信号日后首个交易日开盘买入、T+21 开盘卖出；
- 处理：MAD 去极值、CITIC 2019 一级行业中性化、对数市值中性化、z-score；
- 交易：RQAlpha 开盘撮合、佣金、印花税、最低佣金、涨跌停和成交量上限；
- 评估：IC、RankIC、分组收益、Q5-Q1、HAC/Newey-West、Sharpe、信息比率、回撤与执行审计。

由于本项目是预定义因子的横截面筛选研究，而非训练机器学习模型，采用时间顺序样本内/样本外切分与 HAC 标准误，而非将样本外用于反复调参或 K-fold 优化。

## 3. 因子构造与复现边界

当前等权透明实现固定为四个底层组件各 25%（不是研报原始两层权重公式）：

1. 稳定去杠杆 ROE；
2. 稳定 ROIC 透明代理；
3. 稳定 RONOA 透明代理；
4. 重构 FCFFIC。

其中部分研报原始字段无法逐字段完全获得，因此项目明确使用“严格口径”或“透明代理”标识，不将代理结果表述为对原始研报的逐字逐项完全复现。

严格四组件共同可用样本使综合因子自 2019 年后才具备稳定可检验性；这是本研究的重要覆盖率边界。

## 4. 样本外因子证据

- 样本外平均 RankIC：`{rank_ic:.6f}`；
- RankIC HAC p 值：`{rank_ic_p:.4f}`；
- 样本外平均 Q5-Q1 收益：`{high_low_return:.4%}`；
- Q5-Q1 HAC p 值：`{high_low_p:.4f}`；
- 强制清仓后，Q5 相对 Q1 收益：`{q5_vs_q1:.4%}`；
- 强制清仓后，Q5 相对等权可投资券池收益：`{q5_vs_equal_weight:.4%}`。

RankIC 和 Q5-Q1 虽然在样本外均为正，但 HAC 检验均不显著；Q5 能超过 Q1，却未超过等权可投资券池。因此证据不足以确认独立增量选股 Alpha。

![综合因子有效性](../artifacts/figures/05_integrated_profitability_factor_effectiveness.png)

![样本外年度稳定性](../artifacts/figures/09_integrated_profitability_oos_yearly_stability.png)

## 5. RQAlpha 策略与执行结果

策略规则为：每月选择综合盈利因子最高 20% 的股票、等权做多；持有 20 个交易日；存在重叠持仓时按活跃 sleeves 分配资金。

{strategy_display.to_markdown(index=False)}

Q5 的绝对收益和风险调整表现为正，但等权可投资券池的累计收益与信息比率并不更差。因此，不能把绝对收益误读为因子带来的净选股收益。

![三组合 RQAlpha 对比](../artifacts/figures/08_rqalpha_oos_three_portfolio_comparison.png)

## 6. 失败案例、执行约束与强制清仓检验

RQAlpha 日志显示，部分股票在期末遇到跌停而无法立即卖出。为排除“期末市值计价”对结论的影响，三个组合均在原回测终点后追加 20 个交易日，并在每日设定零目标权重。

结果显示三个组合最终残余市值均为零，完成清仓。强制清仓后，Q5 相对等权券池仍为负值，研究结论不依赖期末未成交头寸。

![强制清仓敏感性检验](../artifacts/figures/09_rqalpha_forced_exit_sensitivity.png)

## 7. 最终研究决策

{decision_display.to_markdown(index=False)}

最终决策：`{final_decision}`。

这是一项有效的负向研究结论：当前等权透明实现具有部分排序信息，但尚不具备作为独立可交易 Alpha 的证据强度。

## 8. 局限与后续方向

当前结论的主要局限包括：

- 综合因子严格共同覆盖率有限；
- RONOA、ROIC 与 FCFFIC 中部分内容为透明代理；
- 尚未进行冲击成本、容量和滑点压力测试；
- 因子在 2023 年出现明显失效，存在市场状态依赖；
- 不应使用已观察到的 2023-2025 样本外结果继续调权或调阈值。

若未来获得新的、未被用于本研究的时间段，可在独立留出样本中预先冻结规则后，检验该候选信号是否在不同市场环境下恢复稳定性。
"""

if REPORT_PATH.exists():
    print("最终报告已存在，未覆盖：", REPORT_PATH)
else:
    REPORT_PATH.write_text(report_text, encoding="utf-8")
    print("最终报告已生成：", REPORT_PATH)

print("\n报告字数：", len(report_text))
print("最终状态：", final_decision)